# 02 CIGNN EMNIST

> EMNIST letter exp

In [1]:
#| default_exp data

%load_ext autoreload
%autoreload 2

In [1]:

import os
import pandas as pd
import sys
sys.path.append(os.path.abspath('..'))

import torch
import torchvision
from torchvision.transforms import ToTensor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision.transforms import functional as F
from torchvision.io import read_image, ImageReadMode
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import numpy as np
import scipy
from sklearn import metrics
import random

from sklearn.metrics import pairwise_distances

import time


In [2]:
project="CIGNN_GA_GoogLeNet_reference"
n_epochs = 100 
train_split = "/data/datasets/HWR/unipen_curated/split/trn.txt"
val_split = "/data/datasets/HWR/unipen_curated/split/val.txt"
test_split = "/data/datasets/HWR/unipen_curated/split/tst.txt"
data_dir = "/data/datasets/HWR/unipen_curated/curated"
model_save_path = "googlenet_unipen_curated.pth"
dev = "cuda"
env = "prod"


In [3]:
# collect all parameters and initialise wandb project
import wandb

wandb.init(
    project=project,
    config = {
        "n_epochs": n_epochs,
        "train_split" : train_split,
        "val_split" : val_split,
        "test_split" : test_split,
        "data_dir" : data_dir,
        "model_save_path" : model_save_path,
        "dev" : dev,
        "env" : env
    }
)



wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: tim-hallyburton (tim-hallyburton-tu-dortmund). Use `wandb login --relogin` to force relogin


In [4]:
def unipen_char_to_code(char: str) -> int: 
    """Convert a single char into the corresponding 0-based index.

    This method respects the fact that UNIPEN (curated) does not have 
    samples for the "\"-symbol. A simple ofsetting -33 for the ASCII
    chars therefore leads to one "extra class", causing indexing errors
    down the line.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2


    Args:
        char (str): The Char "!" - "z" to index.

    Returns:
        int: 0-based index for the input char, skipping the ascii code 92.
    """

    print(char)
    res = ord(char) - 33 
    return res - 1 if (res > (92 - 33)) else res 

def unipen_code_to_char(code: int) -> str: 
    """Convert a 0-based index int to the corresponding ASCII char in the context of the unipen curated dataset.

    This method avoids indexing errors that can happen because the unipen dataset does not provide samples for ascii char 
    92, so simple +-33 conversions cause one empty extra index.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2

    Args:
        code (int): integer of the index to convert to char.

    Returns:
        str: The char after conversion. 
    """

    code = code if code < (92 - 33) else code + 1
    return chr(33 + code)




In [5]:

# load data from curated dataset 
class UnipenCuratedDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None, n_samples=0):
        self.device = torch.device(dev)
        
        self.img_labels = []
        with open(annotations_file, "r") as fh:
            self._img_labels = [[line.strip(), line.strip().split("/")[0]] for line in fh.readlines()]
            self.img_labels = pd.DataFrame(self._img_labels)

        # If the sampling parameter is given, reduce the existing dataframe to the given amount
        if n_samples > 0:
            self.img_labels = self.img_labels.sample(n=min(n_samples, len(self.img_labels)))

        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path, mode=ImageReadMode.RGB)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image.to(torch.float32).to(self.device), int(label) - 33 if int(label) <= 92 else int(label) - 34


In [7]:

train_data = UnipenCuratedDataset(train_split, data_dir) 
train_dataloader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)

val_data = UnipenCuratedDataset(val_split, data_dir) 
val_dataloader = DataLoader(dataset=val_data, batch_size=16, shuffle=True)

test_data = UnipenCuratedDataset(test_split, data_dir)
test_dataloader = DataLoader(dataset=test_data, batch_size=16, shuffle=True)


   
device = torch.device(dev)

model = torchvision.models.googlenet(num_classes=93, aux_logits=False)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

best_val_loss = float('inf')  # Initialize the best validation loss
checkpoint_path = 'best_model_checkpoint.pth'  # Path to save the best model

patience = 3  # Number of epochs to wait for improvement. If we do not increase val performance -> stop 
counter = 0   # Counter for consecutive epochs without improvement (see above)

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    # Training phase
    for batch_idx, (images, labels) in enumerate(train_dataloader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if batch_idx % 10 == 0:
            wandb.log({"train_loss": loss.item()})
            print(f"Epoch [{epoch+1}/{n_epochs}], Batch [{batch_idx}/{len(train_dataloader)}], Loss: {running_loss/batch_idx:.4f}")

    avg_train_loss = running_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Training Loss: {avg_train_loss:.4f}")

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Validation Loss: {avg_val_loss:.4f}")
    wandb.log({"val_loss": avg_val_loss})

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Best model updated and saved at epoch {epoch+1} with Validation Loss: {avg_val_loss:.4f}")
        counter = 0  # Reset counter if performance improves
    else:
        counter += 1
        print(f"No improvement in validation loss for {counter} epoch(s).")
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break
            
# Testing loop -> just calculating accuracy for now
model.load_state_dict(torch.load(checkpoint_path))  # Load the best model checkpoint
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)  # Forward pass
        predicted = outputs.argmax(dim=1)  # Get predicted class
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
wandb.log({"test_acc": accuracy})


/data/thallybu/data/.venv/lib/python3.7/site-packages/torchvision/models/googlenet.py:51: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  FutureWarning,


Epoch [1/100], Batch [10/3158], Loss: 4.5224
Epoch [1/100], Batch [20/3158], Loss: 4.4653
Epoch [1/100], Batch [30/3158], Loss: 4.4042
Epoch [1/100], Batch [40/3158], Loss: 4.3699
Epoch [1/100], Batch [50/3158], Loss: 4.3236
Epoch [1/100], Batch [60/3158], Loss: 4.3096
Epoch [1/100], Batch [70/3158], Loss: 4.2563
Epoch [1/100], Batch [80/3158], Loss: 4.2172
Epoch [1/100], Batch [90/3158], Loss: 4.1668
Epoch [1/100], Batch [100/3158], Loss: 4.1172
Epoch [1/100], Batch [110/3158], Loss: 4.0717
Epoch [1/100], Batch [120/3158], Loss: 4.0353
Epoch [1/100], Batch [130/3158], Loss: 4.0038
Epoch [1/100], Batch [140/3158], Loss: 3.9573
Epoch [1/100], Batch [150/3158], Loss: 3.9082
Epoch [1/100], Batch [160/3158], Loss: 3.8610
Epoch [1/100], Batch [170/3158], Loss: 3.8162
Epoch [1/100], Batch [180/3158], Loss: 3.7737
Epoch [1/100], Batch [190/3158], Loss: 3.7406
Epoch [1/100], Batch [200/3158], Loss: 3.7085
Epoch [1/100], Batch [210/3158], Loss: 3.6701
Epoch [1/100], Batch [220/3158], Loss: 3.62

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()